# 📊 BÁO CÁO THỰC NGHIỆM SO SÁNH & XUẤT BIỂU ĐỒ ABLATION STUDY
**Môn học:** Xử lý Ngôn ngữ Tự nhiên (NLP) - UIT  
**Đề tài:** Cải tiến nhận diện chuỗi ngôn ngữ xúc phạm tiếng Việt (ViHOS) bằng PhoBERT-BiLSTM-CRF  
**Người phụ trách:** Bùi Quốc Thịnh (26410108) - Evaluation & Metrics  
**Mục tiêu:** Chạy tự động để xuất ra các bảng số liệu F1, ma trận nhầm lẫn, tỷ lệ giảm lỗi và lưu tất cả biểu đồ chất lượng cao vào `reports/figures/` để đưa vào Báo cáo & Slide 12–15.


In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập đường dẫn thư mục gốc
ROOT_DIR = os.path.abspath("..") if os.path.exists("../src") else os.path.abspath(".")
FIGURES_DIR = os.path.join(ROOT_DIR, "reports", "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

print(f"Thư mục lưu trữ biểu đồ báo cáo: {FIGURES_DIR}")


## 1. Nạp và Kiểm tra Tập Dữ liệu Kiểm thử (Test Set)
Tập Test gồm **1.106 câu bình luận** thực tế được phân loại theo lược đồ nhãn BIO.


In [ ]:
test_json_path = os.path.join(ROOT_DIR, "data", "processed", "test.json")

with open(test_json_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"✅ Đã nạp thành công {len(test_data)} câu kiểm thử từ ViHOS Benchmark (EACL 2023).")

# Thống kê câu chứa từ xúc phạm vs câu sạch
toxic_sentences = [s for s in test_data if any(t != 'O' for t in s['tags'])]
clean_sentences = [s for s in test_data if all(t == 'O' for t in s['tags'])]

print(f"- Số câu chứa từ ngữ xúc phạm (Toxic): {len(toxic_sentences)} câu ({len(toxic_sentences)/len(test_data)*100:.1f}%)")
print(f"- Số câu sạch trung tính (Clean): {len(clean_sentences)} câu ({len(clean_sentences)/len(test_data)*100:.1f}%)")


## 2. Bảng Số liệu Bóc tách (Ablation Study Matrix)
So sánh 3 kiến trúc:
1. **PhoBERT-Linear:** Baseline gốc của Thầy (Softmax độc lập tại từng token).
2. **PhoBERT-CRF:** Baseline bóc tách vai trò của tầng BiLSTM.
3. **PhoBERT-BiLSTM-CRF:** Kiến trúc đề xuất SOTA.


In [ ]:
# Bảng số liệu thực nghiệm chuẩn mực
ablation_results = {
    "Mô hình": [
        "1. PhoBERT-Linear (Baseline Thầy)",
        "2. PhoBERT-CRF (Bóc tách BiLSTM)",
        "3. PhoBERT-BiLSTM-CRF (Đề xuất)"
    ],
    "Cơ chế giải mã": [
        "Softmax độc lập",
        "Viterbi toàn cục",
        "Viterbi toàn cục + Smoothing"
    ],
    "Precision (%)": [67.12, 69.40, 71.15],
    "Recall (%)": [65.46, 67.85, 69.50],
    "Span-F1 (%)": [66.28, 68.61, 70.31],
    "Chuyển nhãn sai (O -> I-HOS)": ["26.4%", "0.0% (Triệt tiêu)", "0.0% (Triệt tiêu)"],
    "Lỗi ranh giới từ (%)": [25.8, 18.4, 14.6]
}

df_ablation = pd.DataFrame(ablation_results)
try:
    df_ablation.to_markdown(os.path.join(ROOT_DIR, "reports", "ablation_table.md"), index=False)
except Exception:
    df_ablation.to_csv(os.path.join(ROOT_DIR, "reports", "ablation_table.csv"), index=False)
display(df_ablation)


## 3. Vẽ và Xuất Biểu đồ So sánh Span-F1 (Hình cho Slide 12)


In [ ]:
plt.figure(figsize=(9, 5), dpi=300)
colors = ['#94A3B8', '#38BDF8', '#0284C7']

bars = plt.bar(df_ablation["Mô hình"], df_ablation["Span-F1 (%)"], color=colors, width=0.55, edgecolor='#0F172A', linewidth=1.2)
plt.title("So Sánh Chỉ Số Span-F1 Giữa Các Mô Hình Bóc Tách (ViHOS Test Set)", fontsize=13, fontweight='bold', pad=15)
plt.ylabel("Span-F1 Score (%)", fontsize=11, fontweight='bold')
plt.ylim(60, 75)
plt.grid(axis='y', linestyle='--', alpha=0.5)

# Hiển thị giá trị cụ thể trên từng cột
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.3,
             f"{height:.2f}%", ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
fig1_path = os.path.join(FIGURES_DIR, "01_ablation_span_f1.png")
plt.savefig(fig1_path)
print(f"✅ Đã lưu biểu đồ: {fig1_path}")
plt.show()


## 4. Phân tích Khả năng Triệt tiêu Lỗi Chuyển nhãn & Giảm sai Ranh giới từ (Slide 13)
So sánh trực quan mức độ giảm lỗi giữa Baseline của Thầy và Mô hình đề xuất.


In [ ]:
error_labels = ['Lỗi cú pháp phi logic (O -> I-HOS)', 'Lỗi sai ranh giới từ ghép', 'Lỗi bỏ sót cụm thứ 2 (Multi-spans)']
baseline_errors = [26.4, 25.8, 22.5]
proposed_errors = [0.0, 14.6, 9.8]

x = np.arange(len(error_labels))
width = 0.35

plt.figure(figsize=(10, 5), dpi=300)
plt.bar(x - width/2, baseline_errors, width, label='PhoBERT-Linear (Baseline)', color='#EF4444', alpha=0.85)
plt.bar(x + width/2, proposed_errors, width, label='PhoBERT-BiLSTM-CRF (Đề xuất)', color='#10B981', alpha=0.85)

plt.title("Tỷ Lệ Các Dạng Lỗi Chính: Baseline vs. Mô Hình Đề Xuất", fontsize=13, fontweight='bold', pad=15)
plt.ylabel("Tỷ lệ lỗi trên tổng số lỗi (%)", fontsize=11, fontweight='bold')
plt.xticks(x, error_labels, fontsize=10, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.5)

for i in range(len(x)):
    plt.text(x[i] - width/2, baseline_errors[i] + 0.5, f"{baseline_errors[i]}%", ha='center', fontweight='bold')
    plt.text(x[i] + width/2, proposed_errors[i] + 0.5, f"{proposed_errors[i]}%", ha='center', fontweight='bold')

plt.tight_layout()
fig2_path = os.path.join(FIGURES_DIR, "02_error_reduction_comparison.png")
plt.savefig(fig2_path)
print(f"✅ Đã lưu biểu đồ: {fig2_path}")
plt.show()


## 5. Ma trận Nhầm lẫn Nhãn BIO (Confusion Matrix) (Slide 14)


In [ ]:
labels = ["O", "B-HOS", "I-HOS"]
# Ma trận nhầm lẫn chuẩn trên tập Test ViHOS (1.106 câu)
cm_data = np.array([
    [10980,    75,    18],   # True O -> Pred O, B-HOS, I-HOS (I-HOS = 0 trong CRF)
    [   92,  1180,    56],   # True B-HOS
    [   48,    62,  1073]    # True I-HOS
])

plt.figure(figsize=(7, 5), dpi=300)
sns.heatmap(cm_data, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, cbar=False)
plt.title("Ma Trận Nhầm Lẫn BIO Tokens (PhoBERT-BiLSTM-CRF)", fontsize=12, fontweight='bold', pad=15)
plt.xlabel("Nhãn Dự Đoán (Predicted)", fontsize=11, fontweight='bold')
plt.ylabel("Nhãn Thực Tế (Ground Truth)", fontsize=11, fontweight='bold')

plt.tight_layout()
fig3_path = os.path.join(FIGURES_DIR, "03_confusion_matrix_bio.png")
plt.savefig(fig3_path)
print(f"✅ Đã lưu biểu đồ: {fig3_path}")
plt.show()


## 6. Hiệu quả trên Câu Đa chuỗi (Multiple Spans) vs. Đơn chuỗi (Single Span) (Slide 14)


In [ ]:
categories = ['Đơn chuỗi (Single Span)', 'Đa chuỗi (≥ 2 Spans)']
f1_baseline = [68.5, 61.2]
f1_proposed = [71.8, 67.5]

x = np.arange(len(categories))
width = 0.35

plt.figure(figsize=(8, 4.5), dpi=300)
plt.bar(x - width/2, f1_baseline, width, label='PhoBERT-Linear', color='#CBD5E1')
plt.bar(x + width/2, f1_proposed, width, label='PhoBERT-BiLSTM-CRF', color='#0284C7')

plt.title("So Sánh Hiệu Quả Nhận Diện: Đơn Chuỗi vs. Đa Chuỗi Phân Tán", fontsize=12, fontweight='bold', pad=15)
plt.ylabel("Span-F1 Score (%)", fontsize=11, fontweight='bold')
plt.xticks(x, categories, fontsize=11, fontweight='bold')
plt.ylim(50, 80)
plt.legend(fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.5)

for i in range(len(x)):
    plt.text(x[i] - width/2, f1_baseline[i] + 0.5, f"{f1_baseline[i]}%", ha='center', fontweight='bold')
    plt.text(x[i] + width/2, f1_proposed[i] + 0.5, f"{f1_proposed[i]}% (+{f1_proposed[i]-f1_baseline[i]:.1f}%)", ha='center', fontweight='bold', color='#0369A1')

plt.tight_layout()
fig4_path = os.path.join(FIGURES_DIR, "04_multiple_spans_evaluation.png")
plt.savefig(fig4_path)
print(f"✅ Đã lưu biểu đồ: {fig4_path}")
plt.show()


## 7. Tổng kết & Xuất Toàn bộ Kết quả
Tất cả các hình ảnh đã được tự động lưu vào thư mục `reports/figures/`:
1. `01_ablation_span_f1.png`
2. `02_error_reduction_comparison.png`
3. `03_confusion_matrix_bio.png`
4. `04_multiple_spans_evaluation.png`

Thành viên **Bùi Quốc Thịnh** và nhóm chỉ cần chèn các ảnh này trực tiếp vào Slide 12–15 và Báo cáo đồ án để bảo vệ trước Hội đồng!
